# SATF-CNN architecture adapted to the current 8-channel / 3-class SSVEP task

这版不是摘要拼装的 `SATFStyleCNN`，而是按论文正文、图1、表1中明确公开的结构重写。

当前任务适配：
- 论文 9 通道 → 当前 8 通道，所以跨通道卷积 `9×1` 改为 `8×1`。
- 论文 4 类 → 当前 3 类，所以最终 `FC 16→4` 改为 `FC 16→3`。
- 仍严格使用 `test_2~test_7` 训练，validation 只从 source 内部切分。
- `test_8` 和 `four_5` 在模型冻结后才加载。
- 不使用 DANN / GRL / target adaptation。

论文未公开的细节不会伪装成作者参数：
1. Top-k 三分支具体自适应融合公式未给出：本实现用 learnable softmax 标量融合，标记 `PAPER_UNSPECIFIED_ASSUMPTION`。
2. KAN 的 grid / spline order / 层数 / 库未给出：本实现用单层 B-spline KAN fallback，标记 `PAPER_UNSPECIFIED_ASSUMPTION`。
3. Conv2 正文写 padding=23，但表1输出应为15；PyTorch 下 padding=23 得到16，因此本实现用 padding=22 匹配表1，标记 `PAPER_INTERNAL_INCONSISTENCY`。
4. 非整数 Top-k 的取整规则论文未写；当前 C=8 使用 floor，得到 `[4,5,6]`。

In [1]:
from pathlib import Path
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import resample
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cpu


## 1. Strict source-only protocol

In [2]:
DATA_DIR = Path("data")
TRAIN_FILES = [f"test_{i}_ssvep" for i in range(2, 8)]
TEST_FILES = ["test_8_ssvep"]
DRIFT_FILES = ["four_5_ssvep"]

FREQUENCIES_HZ = np.array([15.0, 12.0, 10.0])
NUM_CLASSES = len(FREQUENCIES_HZ)

SAMPLE_RATE_HZ = 250.0
FOUR_DELAY_SECONDS = 0.5
FOUR_WINDOW_SECONDS = 1.5
MODEL_SAMPLES = 372
EEG_CHANNELS = 8
VALIDATION_FRACTION = 0.20

# Paper-confirmed training settings
BATCH_SIZE = 64
MAX_EPOCHS = 500
LEARNING_RATE = 1e-3
LR_PATIENCE = 20

# Current experiment choice, not specified by paper
EARLY_STOP_PATIENCE = 60

TOPK_RATIOS = (0.50, 2 / 3, 0.75)

# PAPER_UNSPECIFIED_ASSUMPTION
KAN_GRID_SIZE = 5
KAN_SPLINE_ORDER = 3

roles = TRAIN_FILES + TEST_FILES + DRIFT_FILES
assert len(roles) == len(set(roles))
assert set(TRAIN_FILES).isdisjoint(TEST_FILES + DRIFT_FILES)
assert set(TEST_FILES).isdisjoint(DRIFT_FILES)

MODEL_FROZEN = False

print("source train/internal val:", TRAIN_FILES)
print("locked source diagnostic:", TEST_FILES)
print("locked final unseen-session:", DRIFT_FILES)
print("target adaptation: NONE")

source train/internal val: ['test_2_ssvep', 'test_3_ssvep', 'test_4_ssvep', 'test_5_ssvep', 'test_6_ssvep', 'test_7_ssvep']
locked source diagnostic: ['test_8_ssvep']
locked final unseen-session: ['four_5_ssvep']
target adaptation: NONE


In [3]:
def prepare_trials(raw, name):
    x = np.stack(raw) if np.asarray(raw).dtype == object else np.asarray(raw)
    if x.ndim != 3 or x.shape[1] != EEG_CHANNELS:
        raise ValueError(f"{name}: expected [trial,{EEG_CHANNELS},time], got {x.shape}")

    if name.startswith("four_"):
        start = round(FOUR_DELAY_SECONDS * SAMPLE_RATE_HZ)
        requested = round(FOUR_WINDOW_SECONDS * SAMPLE_RATE_HZ)
        if x.shape[-1] <= start:
            raise ValueError(f"{name}: cannot skip {start} samples")
        x = x[:, :, start:min(start + requested, x.shape[-1])]
    elif not name.startswith("test_"):
        raise ValueError(f"{name}: unknown session prefix")

    original_samples = x.shape[-1]
    if original_samples != MODEL_SAMPLES:
        x = resample(x, MODEL_SAMPLES, axis=-1)

    x = np.asarray(x, dtype=np.float32)
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    x = (x - mean) / np.maximum(std, 1e-6)
    return x, original_samples


def load_labeled_files(file_names):
    if not MODEL_FROZEN and not set(file_names).issubset(set(TRAIN_FILES)):
        raise RuntimeError("Before freeze, only TRAIN_FILES may be loaded with labels.")

    records = []
    for name in file_names:
        x, original_samples = prepare_trials(
            np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True), name
        )
        y = np.load(DATA_DIR / f"eeg_labels_{name}.npy").astype(np.int64)

        if len(x) != len(y):
            raise ValueError(f"{name}: X/y mismatch")
        if not np.isin(y, np.arange(NUM_CLASSES)).all():
            raise ValueError(f"{name}: labels must be 0..{NUM_CLASSES-1}")

        records.append((x, y, name))
        print("labeled", name, "original", original_samples,
              "shape", x.shape, "counts", np.bincount(y, minlength=NUM_CLASSES))
    return records


def concatenate_labeled(records):
    return (
        np.concatenate([x for x, _, _ in records]),
        np.concatenate([y for _, y, _ in records]),
        np.concatenate([np.full(len(y), name, dtype=object) for _, y, name in records]),
    )

In [4]:
source_records = load_labeled_files(TRAIN_FILES)
X_pool, y_pool, source_pool = concatenate_labeled(source_records)

idx = np.arange(len(y_pool))
strata = np.array([f"{s}::{y}" for s, y in zip(source_pool, y_pool)])

train_idx, val_idx = train_test_split(
    idx,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=strata,
)

assert set(train_idx).isdisjoint(val_idx)

X_train, y_train = X_pool[train_idx], y_pool[train_idx]
X_validation, y_validation = X_pool[val_idx], y_pool[val_idx]

print("source train / validation:", len(y_train), len(y_validation))
print("test_8/four_5 still LOCKED")

labeled test_2_ssvep original 374 shape (19, 8, 372) counts [8 4 7]
labeled test_3_ssvep original 373 shape (41, 8, 372) counts [11 16 14]
labeled test_4_ssvep original 374 shape (69, 8, 372) counts [17 23 29]
labeled test_5_ssvep original 373 shape (102, 8, 372) counts [30 35 37]
labeled test_6_ssvep original 373 shape (106, 8, 372) counts [17 43 46]
labeled test_7_ssvep original 373 shape (67, 8, 372) counts [19 20 28]
source train / validation: 323 81
test_8/four_5 still LOCKED


## 2. Paper sinusoidal position encoding

In [5]:
class PaperSinusoidalPositionEncoding(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        pe = torch.zeros(1, 1, channels, samples, dtype=torch.float32)
        position = torch.arange(samples, dtype=torch.float32)

        pair_index = torch.arange((channels + 1)//2, dtype=torch.float32)
        denominator = torch.pow(10000.0, (2.0 * pair_index) / channels)

        for c in range(channels):
            i = c // 2
            phase = position / denominator[i]
            pe[0, 0, c] = torch.sin(phase) if c % 2 == 0 else torch.cos(phase)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe

## 3. PESA: 1×1 Conv + 3×3 DW-Conv, 3 heads, channel Top-k

In [6]:
class QKVProjection(nn.Module):
    def __init__(self, heads):
        super().__init__()
        self.pointwise = nn.Conv2d(1, heads, kernel_size=1, bias=False)
        self.depthwise = nn.Conv2d(
            heads, heads, kernel_size=3, padding=1, groups=heads, bias=False
        )

    def forward(self, x):
        return self.depthwise(self.pointwise(x))


class PaperPESA(nn.Module):
    def __init__(self, channels, heads=3, ratios=TOPK_RATIOS):
        super().__init__()
        self.channels = channels
        self.heads = heads

        # PAPER_UNSPECIFIED_ASSUMPTION: floor rounding
        self.k_values = tuple(
            max(1, min(channels, math.floor(channels * ratio)))
            for ratio in ratios
        )

        self.q_proj = QKVProjection(heads)
        self.k_proj = QKVProjection(heads)
        self.v_proj = QKVProjection(heads)

        # PAPER_UNSPECIFIED_ASSUMPTION:
        # paper says adaptive fusion but gives no exact implementation.
        self.scale_logits = nn.Parameter(torch.zeros(len(self.k_values)))

        self.out_proj = nn.Conv2d(heads, 1, kernel_size=1, bias=False)

    @staticmethod
    def sparse_attention(scores, values, topk):
        indices = scores.topk(topk, dim=-1).indices
        mask = torch.zeros_like(scores, dtype=torch.bool)
        mask.scatter_(-1, indices, True)
        sparse_scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(sparse_scores, dim=-1)
        return weights @ values, indices

    def forward(self, x, return_indices=False):
        # [B,H,C,T]
        q = F.normalize(self.q_proj(x), p=2, dim=-1)
        k = F.normalize(self.k_proj(x), p=2, dim=-1)
        v = self.v_proj(x)

        # Paper formula uses d=C/Nh.
        d = self.channels / self.heads
        scores = (q @ k.transpose(-1, -2)) / math.sqrt(d)  # [B,H,C,C]

        branches, selections = [], []
        for topk in self.k_values:
            branch, indices = self.sparse_attention(scores, v, topk)
            branches.append(branch)
            selections.append(indices)

        alpha = torch.softmax(self.scale_logits, dim=0)
        fused = sum(w * b for w, b in zip(alpha, branches))

        # Figure 1 shows a residual path.
        out = x + self.out_proj(fused)

        if return_indices:
            return out, selections, scores
        return out

## 4. TCN branch: exact Table-1 shapes

In [7]:
class BSiLU(nn.Module):
    def __init__(self, alpha=1.67):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return (x + self.alpha) * torch.sigmoid(x) - self.alpha / 2.0


class PaperTCN(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(channels,1), bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # PAPER_INTERNAL_INCONSISTENCY:
        # text says p=23 -> PyTorch output 16;
        # table requires output 15, so use p=22.
        self.conv2 = nn.Conv2d(
            16, 16, kernel_size=(1,samples), stride=(1,3), padding=(0,22), bias=False
        )
        self.bn2 = nn.BatchNorm2d(16)

        self.conv3 = nn.Conv2d(16, 16, kernel_size=(1,12), bias=False)
        self.bn3 = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=(1,4), bias=False)
        self.bn4 = nn.BatchNorm2d(32)

        self.act = BSiLU()

        # Paper figure/text: 32-D Conv4 feature -> FC/MLP -> 4-D branch feature.
        self.feature_fc = nn.Linear(32, 4)

    def forward(self, x, return_shapes=False):
        shapes = {}

        x = self.act(self.bn1(self.conv1(x)))
        shapes["conv1"] = tuple(x.shape)

        x = self.act(self.bn2(self.conv2(x)))
        shapes["conv2"] = tuple(x.shape)

        x = self.act(self.bn3(self.conv3(x)))
        shapes["conv3"] = tuple(x.shape)

        x = self.act(self.bn4(self.conv4(x)))
        shapes["conv4"] = tuple(x.shape)

        feature = self.feature_fc(x.flatten(1))
        shapes["feature"] = tuple(feature.shape)

        return (feature, shapes) if return_shapes else feature

## 5. FCN branch: Spatial Conv → 1×1 → FFT → KAN

In [8]:
class SplineKANLinear(nn.Module):
    # PAPER_UNSPECIFIED_ASSUMPTION:
    # paper does not provide KAN grid/order/layer/library.
    def __init__(self, in_features, out_features,
                 grid_size=KAN_GRID_SIZE, spline_order=KAN_SPLINE_ORDER,
                 grid_min=-3.0, grid_max=3.0):
        super().__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order
        self.grid_min = grid_min
        self.grid_max = grid_max

        step = (grid_max - grid_min) / grid_size
        knots = (
            torch.arange(-spline_order, grid_size + spline_order + 1,
                         dtype=torch.float32) * step + grid_min
        )
        self.register_buffer("knots", knots)

        self.base = nn.Linear(in_features, out_features)
        basis_count = grid_size + spline_order
        self.spline_weight = nn.Parameter(
            torch.empty(out_features, in_features, basis_count)
        )
        nn.init.normal_(self.spline_weight, mean=0.0, std=0.02)

    def basis(self, x):
        x = self.grid_max * torch.tanh(x / self.grid_max)
        x = x.unsqueeze(-1)
        k = self.knots

        basis = ((x >= k[:-1]) & (x < k[1:])).to(x.dtype)
        eps = torch.finfo(x.dtype).eps

        for degree in range(1, self.spline_order + 1):
            left_den = (k[degree:-1] - k[:-(degree+1)]).clamp_min(eps)
            right_den = (k[degree+1:] - k[1:-degree]).clamp_min(eps)

            left = ((x - k[:-(degree+1)]) / left_den) * basis[..., :-1]
            right = ((k[degree+1:] - x) / right_den) * basis[..., 1:]
            basis = left + right

        return basis

    def forward(self, x):
        base = self.base(F.silu(x))
        spline = torch.einsum("bin,oin->bo", self.basis(x), self.spline_weight)
        return base + spline


class PaperFCN(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        self.spatial_conv = nn.Conv2d(
            1, 32, kernel_size=(channels,1), bias=False
        )
        self.pointwise = nn.Conv2d(
            32, 16, kernel_size=1, bias=False
        )

        fft_bins = samples // 2 + 1
        self.kan = SplineKANLinear(16 * fft_bins, 4)

    def forward(self, x, return_shapes=False):
        shapes = {}

        x = self.spatial_conv(x)
        shapes["spatial_conv"] = tuple(x.shape)

        x = self.pointwise(x)
        shapes["pointwise"] = tuple(x.shape)

        spectrum = torch.abs(torch.fft.rfft(x, dim=-1))
        shapes["fft"] = tuple(spectrum.shape)

        feature = self.kan(spectrum.flatten(1))
        shapes["feature"] = tuple(feature.shape)

        return (feature, shapes) if return_shapes else feature

## 6. SATF-CNN paper architecture adapted to current task

In [9]:
class SATFCNNPaperAdapted(nn.Module):
    def __init__(self):
        super().__init__()
        self.pe = PaperSinusoidalPositionEncoding(EEG_CHANNELS, MODEL_SAMPLES)
        self.pesa = PaperPESA(EEG_CHANNELS, heads=3)
        self.tcn = PaperTCN(EEG_CHANNELS, MODEL_SAMPLES)
        self.fcn = PaperFCN(EEG_CHANNELS, MODEL_SAMPLES)

        # 4-D time branch + 4-D frequency branch = 8-D.
        self.fusion_fc = nn.Linear(8, 16)
        self.fusion_bn = nn.BatchNorm1d(16)
        self.fusion_act = BSiLU()

        # Task adaptation: paper 16->4; current task 16->3.
        self.classifier = nn.Linear(16, NUM_CLASSES)

    def forward(self, eeg, return_debug=False):
        if eeg.ndim == 3:
            eeg = eeg.unsqueeze(1)
        if eeg.ndim != 4:
            raise ValueError(f"expected [B,C,T] or [B,1,C,T], got {eeg.shape}")

        x = self.pe(eeg)

        if return_debug:
            x, selections, scores = self.pesa(x, return_indices=True)
            tfeat, tshapes = self.tcn(x, return_shapes=True)
            ffeat, fshapes = self.fcn(x, return_shapes=True)
        else:
            x = self.pesa(x)
            tfeat = self.tcn(x)
            ffeat = self.fcn(x)

        fused = torch.cat([tfeat, ffeat], dim=1)
        fused = self.fusion_act(self.fusion_bn(self.fusion_fc(fused)))
        logits = self.classifier(fused)

        if return_debug:
            return {
                "logits": logits,
                "pesa_output_shape": tuple(x.shape),
                "attention_shape": tuple(scores.shape),
                "topk": self.pesa.k_values,
                "selections": selections,
                "tcn": tshapes,
                "fcn": fshapes,
            }
        return logits


model = SATFCNNPaperAdapted().to(DEVICE)
print(model)
print("PESA top-k:", model.pesa.k_values)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

SATFCNNPaperAdapted(
  (pe): PaperSinusoidalPositionEncoding()
  (pesa): PaperPESA(
    (q_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (k_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (v_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (out_proj): Conv2d(3, 1, kernel_size=(1, 1), stride=(1, 1), bias=False)
  )
  (tcn): PaperTCN(
    (conv1): Conv2d(1, 16, kernel_size=(8, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_runn

## 7. Structural forward check only — no synthetic accuracy

In [10]:
# Only tensor-shape verification. No synthetic labels, no accuracy.
model.eval()
with torch.no_grad():
    structural_input = torch.zeros(2, EEG_CHANNELS, MODEL_SAMPLES, device=DEVICE)
    dbg = model(structural_input, return_debug=True)

print("input:", tuple(structural_input.shape))
print("PESA:", dbg["pesa_output_shape"])
print("attention:", dbg["attention_shape"])
print("top-k:", dbg["topk"])
print("TCN:", dbg["tcn"])
print("FCN:", dbg["fcn"])
print("output:", tuple(dbg["logits"].shape))

assert dbg["pesa_output_shape"] == (2,1,EEG_CHANNELS,MODEL_SAMPLES)
assert dbg["attention_shape"] == (2,3,EEG_CHANNELS,EEG_CHANNELS)
assert dbg["tcn"]["conv1"] == (2,16,1,MODEL_SAMPLES)
assert dbg["tcn"]["conv2"] == (2,16,1,15)
assert dbg["tcn"]["conv3"] == (2,16,1,4)
assert dbg["tcn"]["conv4"] == (2,32,1,1)
assert dbg["fcn"]["spatial_conv"] == (2,32,1,MODEL_SAMPLES)
assert dbg["fcn"]["pointwise"] == (2,16,1,MODEL_SAMPLES)
assert dbg["logits"].shape == (2,NUM_CLASSES)

print("shape assertions: PASS")

input: (2, 8, 372)
PESA: (2, 1, 8, 372)
attention: (2, 3, 8, 8)
top-k: (4, 5, 6)
TCN: {'conv1': (2, 16, 1, 372), 'conv2': (2, 16, 1, 15), 'conv3': (2, 16, 1, 4), 'conv4': (2, 32, 1, 1), 'feature': (2, 4)}
FCN: {'spatial_conv': (2, 32, 1, 372), 'pointwise': (2, 16, 1, 372), 'fft': (2, 16, 1, 187), 'feature': (2, 4)}
output: (2, 3)
shape assertions: PASS


## 8. Source-only training

In [11]:
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=LR_PATIENCE
)

@torch.no_grad()
def predict(eeg):
    model.eval()
    x = torch.tensor(eeg, dtype=torch.float32, device=DEVICE)
    return model(x).argmax(1).cpu().numpy()

best_validation_accuracy = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    correct = total = 0

    for batch_eeg, batch_labels in train_loader:
        batch_eeg = batch_eeg.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)

        logits = model(batch_eeg)
        loss = F.cross_entropy(logits, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        correct += (logits.argmax(1) == batch_labels).sum().item()
        total += len(batch_labels)

    train_accuracy = correct / total
    val_pred = predict(X_validation)
    val_accuracy = float(np.mean(val_pred == y_validation))

    scheduler.step(val_accuracy)
    lr = optimizer.param_groups[0]["lr"]

    if val_accuracy > best_validation_accuracy:
        best_validation_accuracy = val_accuracy
        best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 20 == 0:
        print(f"epoch={epoch:03d} train={train_accuracy:.4f} source_val={val_accuracy:.4f} lr={lr:.2e}")

    # Not specified by paper; current strict-source protocol choice.
    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print("early stop at epoch", epoch)
        break

assert best_state is not None
model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()

print("best source validation:", best_validation_accuracy)
print("test_8/four_5 still LOCKED")

epoch=001 train=0.3746 source_val=0.3704 lr=1.00e-03
epoch=020 train=0.9907 source_val=0.8889 lr=1.00e-03
epoch=040 train=1.0000 source_val=0.9136 lr=1.00e-03
epoch=060 train=0.9938 source_val=0.9506 lr=1.00e-03
epoch=080 train=0.9938 source_val=0.9506 lr=5.00e-04
epoch=100 train=0.9969 source_val=0.9506 lr=2.50e-04
epoch=120 train=0.9969 source_val=0.9506 lr=1.25e-04
epoch=140 train=0.9969 source_val=0.9630 lr=6.25e-05
early stop at epoch 151
best source validation: 0.9753086419753086
test_8/four_5 still LOCKED


## 9. Freeze, then load held-out files

In [12]:
for p in model.parameters():
    p.requires_grad_(False)
model.eval()
MODEL_FROZEN = True

print("model frozen; held-out files unlocked")

source_test_records = load_labeled_files(TEST_FILES)
drift_test_records = load_labeled_files(DRIFT_FILES)

X_source_test, y_source_test, _ = concatenate_labeled(source_test_records)
X_drift_test, y_drift_test, _ = concatenate_labeled(drift_test_records)

model frozen; held-out files unlocked
labeled test_8_ssvep original 373 shape (67, 8, 372) counts [23 24 20]
labeled four_5_ssvep original 372 shape (125, 8, 372) counts [25 44 56]


In [13]:
def report(name, labels, pred):
    correct = int(np.sum(pred == labels))
    acc = correct / len(labels)
    bal = balanced_accuracy_score(labels, pred)
    recalls = recall_score(
        labels, pred, labels=np.arange(NUM_CLASSES),
        average=None, zero_division=0
    )
    cm = confusion_matrix(labels, pred, labels=np.arange(NUM_CLASSES))

    print(f"\n{name}")
    print(f"accuracy: {acc:.4f} ({correct}/{len(labels)})")
    print(f"balanced accuracy: {bal:.4f}")
    for f, r in zip(FREQUENCIES_HZ, recalls):
        print(f"recall {f:g} Hz: {r:.4f}")
    print("confusion matrix rows=true [15,12,10], cols=pred [15,12,10]")
    print(cm)

    return acc, bal, recalls, cm

source_pred = predict(X_source_test)
drift_pred = predict(X_drift_test)

report("test_8 source diagnostic", y_source_test, source_pred)
report("four_5 final unseen-session", y_drift_test, drift_pred)


test_8 source diagnostic
accuracy: 0.8806 (59/67)
balanced accuracy: 0.8859
recall 15 Hz: 0.7826
recall 12 Hz: 0.8750
recall 10 Hz: 1.0000
confusion matrix rows=true [15,12,10], cols=pred [15,12,10]
[[18  1  4]
 [ 1 21  2]
 [ 0  0 20]]

four_5 final unseen-session
accuracy: 0.4080 (51/125)
balanced accuracy: 0.4329
recall 15 Hz: 0.5600
recall 12 Hz: 0.3636
recall 10 Hz: 0.3750
confusion matrix rows=true [15,12,10], cols=pred [15,12,10]
[[14  7  4]
 [16 16 12]
 [21 14 21]]


(0.408,
 0.4328787878787879,
 array([0.56      , 0.36363636, 0.375     ]),
 array([[14,  7,  4],
        [16, 16, 12],
        [21, 14, 21]]))

## 10. PESA selections on final trials

In [14]:
sample = torch.tensor(X_drift_test[:3], dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    dbg = model(sample, return_debug=True)

for topk, indices in zip(dbg["topk"], dbg["selections"]):
    indices = indices.cpu().numpy()
    print(f"top-{topk} shape={indices.shape}")
    for trial_idx in range(len(sample)):
        print(f"  trial {trial_idx+1}: {indices[trial_idx].tolist()}")

top-4 shape=(3, 3, 8, 4)
  trial 1: [[[1, 3, 5, 6], [0, 2, 6, 3], [3, 1, 0, 5], [2, 0, 6, 7], [3, 7, 0, 5], [4, 7, 0, 2], [5, 7, 0, 2], [7, 6, 4, 2]], [[2, 5, 7, 4], [3, 2, 7, 5], [4, 7, 3, 5], [5, 7, 1, 4], [6, 5, 2, 7], [7, 3, 4, 0], [7, 4, 5, 1], [0, 6, 1, 3]], [[0, 2, 5, 3], [0, 1, 2, 5], [0, 2, 1, 5], [0, 3, 2, 1], [0, 4, 2, 1], [0, 2, 1, 5], [0, 6, 2, 4], [0, 6, 4, 1]]]
  trial 2: [[[1, 6, 4, 5], [0, 2, 6, 4], [0, 1, 3, 6], [2, 0, 6, 4], [3, 7, 0, 6], [4, 7, 0, 6], [5, 7, 3, 0], [7, 6, 4, 0]], [[2, 1, 7, 5], [3, 2, 7, 5], [4, 7, 3, 5], [5, 7, 1, 4], [6, 5, 2, 7], [7, 4, 3, 1], [7, 4, 5, 1], [0, 6, 3, 2]], [[0, 6, 2, 4], [0, 1, 2, 6], [0, 2, 5, 1], [0, 3, 2, 1], [0, 4, 1, 3], [0, 5, 3, 1], [0, 6, 3, 4], [0, 6, 3, 4]]]
  trial 3: [[[1, 5, 3, 7], [0, 2, 7, 5], [1, 3, 0, 5], [2, 0, 7, 5], [3, 5, 7, 0], [4, 7, 2, 0], [5, 7, 3, 0], [7, 6, 4, 2]], [[2, 1, 7, 4], [3, 7, 5, 2], [4, 3, 7, 5], [5, 7, 1, 4], [6, 5, 2, 7], [7, 3, 4, 0], [7, 4, 5, 1], [0, 1, 6, 3]], [[0, 2, 5, 4], [0, 1, 2, 5]

## Reporting boundary

可以称：
- `SATF-CNN architecture adapted to our 8-channel 3-class task`

不能称：
- 作者官方代码
- 逐参数完全复现
- 已复现论文 93.54%

当前仍有两个论文未公开实现细节：
- PESA 三个 Top-k 稀疏尺度的具体自适应融合实现
- KAN 的具体实现与参数

一旦获得作者代码，只替换这两个 `PAPER_UNSPECIFIED_ASSUMPTION` 即可。